# 감성분류 모델 학습 — 하이브리드 프로젝트 베이스라인 (★★★)

**NSMC(네이버 영화리뷰)** 데이터로 긍정/부정 분류 모델을 학습해 저장합니다.
저장된 모델은 `server.py`가 로드해서 **"내 모델이 분류 → LLM이 코멘트 생성"** 파이프라인의 앞단을 맡습니다.

> 완성 기준(2일차): 테스트 정확도 80% 이상 + `sentiment.pt` / `vectorizer.pkl` 저장 + 실험 기록 2건

구조: 텍스트 → **TF-IDF 벡터화**(sklearn) → **작은 MLP**(PyTorch) → 긍정/부정

⚠️ 전처리기(vectorizer)와 모델은 **한 몸**입니다. 서빙할 때도 같은 vectorizer를 써야 하므로 둘 다 저장합니다.

## 1. 데이터 내려받기 (NSMC, 처음 한 번만 몇 초 소요)

In [2]:
import pandas as pd

TRAIN_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
TEST_URL  = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"

train_df = pd.read_csv(TRAIN_URL, sep="\t").dropna(subset=["document"])
test_df  = pd.read_csv(TEST_URL,  sep="\t").dropna(subset=["document"])

# 빠른 실험을 위해 서브셋 사용 — TODO: 늘리면 정확도가 오르는지 확인
train_df = train_df.sample(n=15000, random_state=42)
test_df  = test_df.sample(n=5000,  random_state=42)

print("train:", len(train_df), "/ test:", len(test_df))
train_df.head()

train: 15000 / test: 5000


,id,document,label
128076,7865795,원본이 최고,1
65517,5417631,스릴감과 훈훈함이 있는 영화.,1
85048,8357466,굉장히 저평가되는 영화중 하나라고 생각함,1
118192,8252946,정말영화같은이야기 영화여서 영화같은이야기가 좋다,1
92368,7800452,계기도없는데 이상하다,0


## 2. TF-IDF 벡터화

한국어 형태소 분석기 없이도 동작하도록 **문자 n-gram** 방식을 씁니다.
`max_features`가 모델 입력 차원이 됩니다.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="char_wb",      # 문자 단위 n-gram (한국어에 무난)
    ngram_range=(1, 3),
    max_features=2000,        # TODO: 5000으로 늘리면? (메모리 주의)
)

X_train = vectorizer.fit_transform(train_df["document"]).toarray().astype("float32")
X_test  = vectorizer.transform(test_df["document"]).toarray().astype("float32")
y_train = train_df["label"].values
y_test  = test_df["label"].values

print("X_train:", X_train.shape, "/ X_test:", X_test.shape)

X_train: (15000, 2000) / X_test: (5000, 2000)


## 3. 모델 학습 (PyTorch MLP)

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from model import SentimentMLP

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_ds  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

model = SentimentMLP(input_dim=X_train.shape[1]).to(device)

EPOCHS = 10     # TODO: 늘리거나 줄여보기
LR = 1e-3      # TODO: 바꿔보기

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1} | 평균 loss {total_loss/len(train_loader):.4f}")

epoch 1 | 평균 loss 0.5468
epoch 2 | 평균 loss 0.4003
epoch 3 | 평균 loss 0.3783
epoch 4 | 평균 loss 0.3632
epoch 5 | 평균 loss 0.3526
epoch 6 | 평균 loss 0.3403
epoch 7 | 평균 loss 0.3282
epoch 8 | 평균 loss 0.3182
epoch 9 | 평균 loss 0.3054
epoch 10 | 평균 loss 0.2931


## 4. 평가 — 이 숫자가 KPI입니다

In [8]:
model.eval()
correct = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(dim=1) == y).sum().item()

accuracy = correct / len(test_ds)
print(f"테스트 정확도: {accuracy:.4f} ({correct}/{len(test_ds)})")

테스트 정확도: 0.8036 (4018/5000)


## 5. 모델 + 벡터라이저 저장 (둘 다!)

`server.py`가 시작할 때 두 파일을 모두 로드합니다.

In [9]:
import joblib

torch.save(
    {"state_dict": model.state_dict(), "input_dim": X_train.shape[1]},
    "sentiment.pt",
)
joblib.dump(vectorizer, "vectorizer.pkl")
print("sentiment.pt / vectorizer.pkl 저장 완료")

sentiment.pt / vectorizer.pkl 저장 완료


In [10]:
# 서버가 하는 일과 동일하게 다시 불러와 예측 확인
ckpt = torch.load("sentiment.pt", map_location="cpu")
loaded = SentimentMLP(input_dim=ckpt["input_dim"])
loaded.load_state_dict(ckpt["state_dict"])
loaded.eval()
vec = joblib.load("vectorizer.pkl")

from model import LABELS

for text in ["오늘 발표 완전 망했다... 최악이야", "드디어 프로젝트 완성! 너무 뿌듯하다"]:
    x = torch.tensor(vec.transform([text]).toarray().astype("float32"))
    with torch.no_grad():
        probs = torch.softmax(loaded(x), dim=1)[0]
    print(f"{text!r} → {LABELS[int(probs.argmax())]} ({float(probs.max()):.2f})")

'오늘 발표 완전 망했다... 최악이야' → 부정 (1.00)
'드디어 프로젝트 완성! 너무 뿌듯하다' → 부정 (0.54)


---

## TODO — 하나 바꿀 때마다 기록!

- [ ] TODO 1. 학습 데이터 15,000 → 50,000으로 늘리기
- [ ] TODO 2. `max_features` 2000 → 5000 (정확도 vs 학습시간 트레이드오프 기록)
- [ ] TODO 3. `EPOCHS` / `LR` 튜닝
- [ ] TODO 4. (심화) 틀린 문장 10개를 뽑아 유형 분석 — 반어법? 짧은 문장?
- [ ] TODO 5. (심화) 같은 문장을 LLM에게도 분류시켜 **내 모델 vs LLM** 정확도·속도 비교

## 실험 기록 (발표 자료에 그대로 들어갑니다)

| # | 바꾼 것 | 테스트 정확도 | 학습 시간 | 메모 |
|---|---|---|---|---|
| 0 | baseline (15k, feat 2000, epoch 5) | 0.____ |  |  |
| 1 |  |  |  |  |
| 2 |  |  |  |  |

다 됐으면 → `README.md` 순서대로 서버(`server.py`)와 UI(`app.py`)를 띄워 파이프라인을 완성하세요.